# cycle-detection-temp-set — worked example 2: Raise a ValueError naming the offending node on cycle

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cycle-detection-temp-set`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

This is the sibling response to the boolean version: instead of returning a flag, the traversal *raises* when it re-enters a vertex in the gray (`temp`) set. The distinction is identical — `perm` means 'done, return quietly'; `temp` means 'still on the stack, this is a back-edge, raise'. Backprop topological sort uses exactly this raise-on-gray behaviour to refuse to schedule a computation graph that contains a cycle.

## Worked solution

We must topologically order vertices of `adj`, raising `ValueError` if a cycle blocks the ordering, and the message must name the node where the back-edge closed.

**Step 1 — three states.** `perm` = finished, `temp` = on stack. Anything in neither is unvisited. The output `order` list is appended to as subtrees finish.

**Step 2 — gray re-entry raises.** In `visit(u)`: if `u in perm`, return (already placed). If `u in temp`, we have hit a vertex still on the recursion stack — raise `ValueError(f"cycle through {u}")`. This is the only place the algorithm rejects the graph.

**Step 3 — gray, recurse, black, emit.** Add `u` to `temp`, recurse into neighbours, then `temp.remove(u)`, `perm.add(u)`, and `order.append(u)`. Appending *after* the children are finished produces a reverse-postorder; reversing at the end gives a valid topological order (every edge points forward).

**Step 4 — drive every component.** We loop over all keys so disconnected pieces are covered, only starting a DFS from still-unvisited vertices.

**Why it's correct.** Re-entering a gray vertex is the back-edge condition, the exact signature of a directed cycle, so raising there is sound. Because finished vertices are appended after all their descendants, reversing the postorder respects every edge — the defining property of a topo sort.

In [ ]:
def toposort(adj):
    perm, temp, order = set(), set(), []

    def visit(u):
        if u in perm:
            return
        if u in temp:
            raise ValueError(f"cycle through {u}")
        temp.add(u)
        for v in adj.get(u, []):
            visit(v)
        temp.remove(u)
        perm.add(u)
        order.append(u)

    for u in adj:
        visit(u)
    order.reverse()
    return order


dag = {"x": ["y", "z"], "y": ["z"], "z": []}
print("topo order:", toposort(dag))
try:
    toposort({"p": ["q"], "q": ["p"]})
except ValueError as e:
    print("raised:", e)